# Python para Analista de Dados — do SQL ao pandas

**Para quem já escreve SQL e quer fazer a mesma análise em Python.**

Cada conceito aparece em duas versões: a consulta SQL que você já sabe escrever e a
linha de pandas equivalente. A base é `Base_Vendas_Exploracao.xlsx` — 10.000 vendas
de varejo em 24 meses, com cliente, produto, região, vendedor, custo e margem.

**Como usar:** rode as células de cima para baixo (`Shift + Enter`). Coloque o arquivo
`.xlsx` na mesma pasta deste notebook.

---
## Módulo 0 — As bibliotecas que importam

Você não precisa aprender "Python". Precisa aprender **quatro bibliotecas**.

| Biblioteca | Apelido | O que faz | O equivalente mental |
|---|---|---|---|
| **pandas** | `pd` | Tabelas: ler, filtrar, agrupar, juntar, pivotar | O SQL + o Excel |
| **numpy** | `np` | Cálculo vetorizado, arrays, `np.where`, estatística | A calculadora por trás do pandas |
| **matplotlib** | `plt` | Gráficos, com controle total de cada detalhe | O motor de gráficos |
| **seaborn** | `sns` | Gráficos estatísticos prontos em uma linha | Um atalho por cima do matplotlib |

Três que aparecem sem você chamar:

| Biblioteca | Para quê |
|---|---|
| **openpyxl** | O pandas usa por baixo para ler e escrever `.xlsx` |
| **pyarrow** | Formato Parquet — muito mais rápido que CSV para base grande |
| **scikit-learn** | Modelagem: regressão, árvore, clusterização, métricas |

**Regra prática:** 90% do trabalho de um analista júnior é pandas. Aprenda pandas fundo
antes de olhar para o resto.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# deixa a saída legível no notebook
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)

print("pandas", pd.__version__, "| numpy", np.__version__)

---
## Módulo 1 — Ler a base e olhar para ela

No banco você daria `SELECT TOP 10 * FROM vendas`. Aqui você **carrega a tabela inteira na
memória** e ela vira um objeto chamado **DataFrame**.

Dois nomes que você vai ouvir o tempo todo:

- **DataFrame** — a tabela inteira (linhas × colunas). É o `df`.
- **Series** — uma coluna só. `df["Valor_Venda"]` é uma Series.

In [ ]:
ARQUIVO = "Base_Vendas_Exploracao.xlsx"

df = pd.read_excel(ARQUIVO, sheet_name="Vendas")

print("linhas x colunas:", df.shape)
df.head(3)

Os quatro comandos de reconhecimento que você roda **sempre** ao receber uma base nova:

In [ ]:
df.info()          # tipos de cada coluna e quantos nulos existem

In [ ]:
df.describe().T    # estatística das colunas numéricas: media, desvio, min, quartis, max

In [ ]:
# quantos valores distintos cada coluna de texto tem
df.select_dtypes(exclude="number").nunique().sort_values()

In [ ]:
# nulos por coluna — aqui a base está limpa, mas na vida real quase nunca está
df.isna().sum().loc[lambda s: s > 0]

---
## Módulo 2 — SELECT e WHERE

### Escolher colunas

```sql
SELECT Cliente, Produto, Valor_Venda FROM vendas
```

In [ ]:
df[["Cliente", "Produto", "Valor_Venda"]].head()

Atenção ao **colchete duplo**: `df["Cliente"]` devolve uma Series (uma coluna);
`df[["Cliente"]]` devolve um DataFrame de uma coluna. A lista dentro do colchete é
que faz a diferença.

### Filtrar linhas

```sql
SELECT * FROM vendas WHERE Regiao = 'Sudeste' AND Valor_Venda > 5000
```

Em pandas o filtro é uma **máscara booleana** — uma Series de True/False do mesmo
tamanho da tabela, que você usa para fatiar.

In [ ]:
mask = (df["Regiao"] == "Sudeste") & (df["Valor_Venda"] > 5000)
print("tipo da máscara:", type(mask).__name__, "| quantos True:", mask.sum())

df[mask].head(3)

**Três armadilhas que pegam todo mundo que vem do SQL:**

1. O operador é `&` e `|`, **não** `and` / `or`.
2. Cada condição precisa de parênteses próprios: `(a == 1) & (b > 2)`.
3. Comparação de igualdade é `==`, com dois sinais.

### `.query()` — quando você quer que pareça SQL

Mais legível para filtros longos, e não exige parênteses em cada condição:

In [ ]:
df.query("Regiao == 'Sudeste' and Valor_Venda > 5000 and Status_Pedido == 'Concluida'").shape

### O resto do WHERE

| SQL | pandas |
|---|---|
| `IN ('A','B')` | `df["col"].isin(["A", "B"])` |
| `BETWEEN 10 AND 20` | `df["col"].between(10, 20)` |
| `LIKE '%Note%'` | `df["col"].str.contains("Note")` |
| `IS NULL` | `df["col"].isna()` |
| `NOT ...` | `~(...)` — o til inverte a máscara |

In [ ]:
filtro = (
    df["Categoria"].isin(["Eletronicos", "Moveis"])
    & df["Desconto_Pct"].between(0.05, 0.20)
    & df["Subproduto"].str.contains("Note")
    & ~df["Status_Pedido"].isin(["Cancelada"])
)
df[filtro][["Subproduto", "Desconto_Pct", "Valor_Venda"]].head()

### `.loc` — filtrar linhas e escolher colunas na mesma linha de código

`.loc[linhas, colunas]` é a forma canônica. Use sempre que for filtrar **e** selecionar.

In [ ]:
df.loc[df["Margem_Pct"] < 0, ["Data_Venda", "Subproduto", "Desconto_Pct", "Margem_Pct"]].head()

---
## Módulo 3 — ORDER BY e LIMIT

```sql
SELECT * FROM vendas ORDER BY Valor_Venda DESC LIMIT 5
```

In [ ]:
df.sort_values("Valor_Venda", ascending=False).head(5)[["Cliente", "Subproduto", "Valor_Venda"]]

Ordenar por mais de uma coluna, com direções diferentes — igual ao SQL:

In [ ]:
df.sort_values(["Regiao", "Valor_Venda"], ascending=[True, False]) \
  [["Regiao", "Cliente", "Valor_Venda"]].head(6)

**Atalho:** para "os N maiores" existe `nlargest`, que é mais rápido e mais direto que
ordenar a tabela toda.

In [ ]:
df.nlargest(5, "Valor_Venda")[["Subproduto", "Valor_Venda"]]

---
## Módulo 4 — GROUP BY: o coração da análise

```sql
SELECT Regiao, SUM(Valor_Venda) AS receita, COUNT(*) AS pedidos
FROM vendas GROUP BY Regiao ORDER BY receita DESC
```

O `groupby` do pandas faz exatamente isso. A diferença é que o resultado do agrupamento
vira o **índice** da tabela, não uma coluna comum.

In [ ]:
df.groupby("Regiao")["Valor_Venda"].sum().sort_values(ascending=False)

### Várias métricas ao mesmo tempo: `.agg()` com nomes

Esta é a forma que você vai usar 90% das vezes. Cada linha do `agg` é
`nome_da_saida=("coluna_de_origem", "funcao")`.

In [ ]:
resumo = (
    df.groupby("Regiao")
      .agg(
          pedidos=("ID_Venda", "count"),
          receita=("Valor_Venda", "sum"),
          ticket_medio=("Valor_Venda", "mean"),
          margem=("Margem_Valor", "sum"),
      )
      .assign(margem_pct=lambda d: d["margem"] / d["receita"])
      .sort_values("receita", ascending=False)
)
resumo.round(2)

Repare no `.assign()`: ele cria uma coluna nova a partir das que acabaram de ser
calculadas, sem quebrar a cadeia. O `lambda d:` é "pegue a tabela que chegou até aqui".

### Agrupar por mais de uma coluna

```sql
GROUP BY Regiao, Categoria
```

In [ ]:
por_reg_cat = df.groupby(["Regiao", "Categoria"])["Valor_Venda"].sum()
por_reg_cat.head(8)

### `pivot_table` — o GROUP BY que vira tabela dinâmica

Quando você quer uma dimensão nas linhas e outra nas colunas, `pivot_table` é mais
legível que `groupby` + `unstack`. É literalmente a tabela dinâmica do Excel.

In [ ]:
pv = df.pivot_table(
    index="Regiao",
    columns="Categoria",
    values="Valor_Venda",
    aggfunc="sum",
    margins=True,          # a linha e a coluna de total
    margins_name="Total",
)
(pv / 1_000_000).round(2)   # em R$ milhões, para caber na tela

### HAVING

No SQL o `HAVING` filtra **depois** de agregar. Em pandas você simplesmente filtra o
resultado do groupby — porque ele já é uma tabela.

```sql
... GROUP BY Vendedor HAVING SUM(Valor_Venda) > 4000000
```

In [ ]:
por_vendedor = df.groupby("Vendedor")["Valor_Venda"].sum()
por_vendedor[por_vendedor > 4_000_000].sort_values(ascending=False)

### `reset_index()` — o comando que você vai esquecer e depois nunca mais

Depois de um groupby, a chave vira índice. Para voltar a ter uma coluna normal
(e conseguir plotar, exportar ou juntar com outra tabela), use `reset_index()`.

In [ ]:
tabela = df.groupby("Canal_Venda")["Valor_Venda"].sum().reset_index()
tabela.columns = ["Canal_Venda", "Receita"]
tabela.sort_values("Receita", ascending=False)

---
## Módulo 5 — JOIN

```sql
SELECT v.*, m.Meta_Mensal
FROM vendas v LEFT JOIN metas m ON v.Regiao = m.Regiao
```

Em pandas o JOIN se chama **`merge`**. A tradução é direta:

| SQL | pandas |
|---|---|
| `INNER JOIN` | `how="inner"` (o padrão) |
| `LEFT JOIN` | `how="left"` |
| `FULL OUTER JOIN` | `how="outer"` |
| `ON a.x = b.y` | `left_on="x", right_on="y"` |

Vamos criar uma tabela de metas por região — o seu "de-para" — e juntar.

In [ ]:
metas = pd.DataFrame({
    "Regiao": ["Sudeste", "Sul", "Nordeste", "Centro-Oeste", "Norte"],
    "Meta_Anual": [12_000_000, 6_000_000, 7_000_000, 2_800_000, 1_900_000],
    "Gerente": ["Paula Vasques", "Rui Antunes", "Sonia Belmiro", "Tarcisio Melo", "Ubirajara Pinho"],
})
metas

In [ ]:
receita_2025 = (
    df[df["Data_Venda"].dt.year == 2025]
      .groupby("Regiao", as_index=False)["Valor_Venda"].sum()
      .rename(columns={"Valor_Venda": "Receita_2025"})
)

painel = receita_2025.merge(metas, on="Regiao", how="left", validate="one_to_one")
painel["Atingimento"] = painel["Receita_2025"] / painel["Meta_Anual"]
painel.sort_values("Atingimento", ascending=False).round(3)

**Dois argumentos que salvam a sua análise — use sempre:**

- `validate="one_to_one"` (ou `"many_to_one"`) faz o pandas **quebrar com erro** se a
  cardinalidade não for a que você espera. Sem isso, um de-para com chave duplicada
  multiplica linhas silenciosamente e a sua receita infla. É o erro nº 1 de analista júnior.
- `indicator=True` cria uma coluna dizendo se a linha veio das duas tabelas ou só de uma —
  perfeito para auditar o que não casou.

In [ ]:
teste = receita_2025.merge(metas, on="Regiao", how="outer", indicator=True)
teste["_merge"].value_counts()

### `concat` — o UNION

`merge` junta **lado a lado** (colunas). `concat` empilha **uma embaixo da outra** (linhas).
É o `UNION ALL`.

In [ ]:
a = df[df["Regiao"] == "Norte"].head(2)
b = df[df["Regiao"] == "Sul"].head(2)
pd.concat([a, b], ignore_index=True)[["Regiao", "Cliente", "Valor_Venda"]]

---
## Módulo 6 — Datas

Datas são a fonte mais comum de erro silencioso. A regra: garanta que a coluna é do tipo
`datetime` **antes** de qualquer coisa. Depois disso, tudo vem pelo acessório `.dt`.

In [ ]:
df["Data_Venda"] = pd.to_datetime(df["Data_Venda"])   # se já for datetime, não faz mal
print(df["Data_Venda"].dtype)

partes = pd.DataFrame({
    "data": df["Data_Venda"].head(4),
    "ano": df["Data_Venda"].dt.year.head(4),
    "mes": df["Data_Venda"].dt.month.head(4),
    "trimestre": df["Data_Venda"].dt.quarter.head(4),
    "dia_semana": df["Data_Venda"].dt.day_name().head(4),
})
partes

### Agrupar por mês: `to_period`

```sql
GROUP BY DATE_TRUNC('month', Data_Venda)
```

In [ ]:
mensal = df.groupby(df["Data_Venda"].dt.to_period("M"))["Valor_Venda"].sum()
mensal.head(6)

### Filtrar por período

Com a data como índice, você filtra por texto — `"2025"`, `"2025-11"`, ou uma fatia.

In [ ]:
por_dia = df.set_index("Data_Venda")["Valor_Venda"].resample("D").sum()
print("receita de novembro/2025: R$ %.0f" % por_dia["2025-11"].sum())
print("receita do 4o trimestre/2025: R$ %.0f" % por_dia["2025-10":"2025-12"].sum())

`resample` é o groupby exclusivo de datas: `"D"` diário, `"W"` semanal, `"M"` mensal,
`"Q"` trimestral, `"Y"` anual.

### Comparar com o mesmo mês do ano anterior

`shift(12)` empurra a série 12 posições — é o `LAG(coluna, 12)` das funções de janela.

In [ ]:
comp = mensal.to_frame("receita")
comp["ano_anterior"] = comp["receita"].shift(12)
comp["var_pct"] = comp["receita"] / comp["ano_anterior"] - 1
comp.dropna().head(6).round(3)

---
## Módulo 7 — Criar colunas e o CASE WHEN

```sql
SELECT *, CASE WHEN Valor_Venda > 5000 THEN 'Alto' ELSE 'Baixo' END AS faixa FROM vendas
```

Três ferramentas, da mais simples para a mais completa.

In [ ]:
# 1) duas saídas -> np.where (o IF do Excel)
df["porte"] = np.where(df["Valor_Venda"] > 5000, "Alto", "Baixo")
df["porte"].value_counts()

In [ ]:
# 2) faixas numéricas -> pd.cut (o PROCV aproximado / faixas de score)
df["faixa_desconto"] = pd.cut(
    df["Desconto_Pct"],
    bins=[-0.001, 0.05, 0.10, 0.20, 1.0],
    labels=["Sem desconto ate 5%", "5-10%", "10-20%", "Acima de 20%"],
)
df.groupby("faixa_desconto", observed=True)["Margem_Pct"].mean().round(3)

In [ ]:
# 3) várias condições encadeadas -> np.select (o CASE WHEN completo)
condicoes = [
    df["Margem_Pct"] < 0,
    df["Margem_Pct"] < 0.15,
    df["Margem_Pct"] < 0.30,
]
resultados = ["Prejuizo", "Margem baixa", "Margem media"]
df["classe_margem"] = np.select(condicoes, resultados, default="Margem alta")

df["classe_margem"].value_counts()

`np.select` avalia as condições **na ordem** e para na primeira verdadeira — igual ao
`CASE WHEN`. O `default` é o `ELSE`.

---
## Módulo 8 — Visualização

Todo gráfico responde a uma pergunta. Escolha a forma pela pergunta, não pelo que é bonito:

| A pergunta | A forma |
|---|---|
| Como os valores se espalham? | **Histograma** ou **boxplot** |
| Como evoluiu no tempo? | **Linha** |
| Quem é maior? | **Barra horizontal**, ordenada |
| Duas medidas andam juntas? | **Dispersão** |
| Quanto é o total? | Às vezes **não é gráfico** — é um número grande |

### O gráfico mínimo

O pandas plota direto, sem nenhuma configuração. Serve para você olhar rápido enquanto
analisa:

In [ ]:
df["Valor_Venda"].plot(kind="hist", bins=40, title="Rascunho: distribuicao do valor")
plt.show()

### O gráfico apresentável

O rascunho serve para você. Para mandar para alguém, vale aplicar um estilo. Defina isto
**uma vez** no começo do notebook e todos os gráficos seguintes herdam:

In [ ]:
import matplotlib.ticker as mticker

PALETA = {
    "surface": "#fcfcfb", "ink": "#0b0b0b", "ink2": "#52514e",
    "muted": "#898781", "grid": "#e1e0d9", "axis": "#c3c2b7",
    "azul": "#2a78d6", "laranja": "#eb6834",
}

plt.rcParams.update({
    "font.size": 11,
    "figure.facecolor": PALETA["surface"], "axes.facecolor": PALETA["surface"],
    "savefig.facecolor": PALETA["surface"],
    "text.color": PALETA["ink"], "axes.labelcolor": PALETA["ink2"],
    "xtick.color": PALETA["muted"], "ytick.color": PALETA["muted"],
    "axes.edgecolor": PALETA["axis"], "grid.color": PALETA["grid"], "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False,
})

def acabamento(ax, titulo, subtitulo=None, grid="y"):
    """Aplica titulo, subtitulo e grade discreta. Chame no fim de cada grafico."""
    getattr(ax, f"{grid}axis").grid(True, color=PALETA["grid"], linewidth=0.8)
    ax.set_axisbelow(True)
    ax.set_title(titulo, color=PALETA["ink"], fontsize=13, fontweight="bold",
                 loc="left", pad=18 if subtitulo else 10)
    if subtitulo:
        ax.text(0, 1.02, subtitulo, transform=ax.transAxes,
                color=PALETA["ink2"], fontsize=10, va="bottom")

def brl(x, pos=None):
    """Formata o eixo em reais."""
    if abs(x) >= 1_000_000:
        return f"R$ {x/1_000_000:,.1f} mi".replace(",", "X").replace(".", ",").replace("X", ".")
    if abs(x) >= 1_000:
        return f"R$ {x/1_000:,.0f} mil".replace(",", ".")
    return f"R$ {x:,.0f}".replace(",", ".")

print("estilo aplicado")

#### 8.1 Histograma — como os valores se espalham

Corte o percentil 99 antes de plotar. Sem isso, meia dúzia de vendas gigantes espremem
todo o resto contra a esquerda e o gráfico não mostra nada.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.4))

dados = df.loc[df["Valor_Venda"] < df["Valor_Venda"].quantile(0.99), "Valor_Venda"]
ax.hist(dados, bins=45, color=PALETA["azul"], edgecolor=PALETA["surface"], linewidth=1.2)

mediana = df["Valor_Venda"].median()
ax.axvline(mediana, color=PALETA["laranja"], linewidth=2)
ax.text(mediana, ax.get_ylim()[1] * 0.92, f"  mediana: {brl(mediana)}",
        color=PALETA["laranja"], fontsize=10, fontweight="bold", va="top")

ax.xaxis.set_major_formatter(mticker.FuncFormatter(brl))
ax.set_xlabel("Valor da venda"); ax.set_ylabel("Numero de transacoes")
acabamento(ax, "Distribuicao do valor das vendas",
           "Assimetrica a direita: muitas vendas pequenas, poucas muito grandes")
plt.show()

**Como ler:** a distribuição é assimétrica à direita. Por isso a **mediana** descreve
melhor "a venda típica" do que a média — a média é puxada para cima pelas poucas vendas
enormes. Sempre que o histograma tiver essa cara, prefira mediana.

#### 8.2 Linha — evolução no tempo

In [ ]:
import matplotlib.dates as mdates

serie = df.groupby(df["Data_Venda"].dt.to_period("M"))["Valor_Venda"].sum()
x = serie.index.to_timestamp()

fig, ax = plt.subplots(figsize=(8, 4.4))
ax.plot(x, serie.values, color=PALETA["azul"], linewidth=2)

i_max, i_min = serie.values.argmax(), serie.values.argmin()
for i, cor, dy, txt in [(i_max, PALETA["azul"], 12, "pico"), (i_min, PALETA["laranja"], -18, "vale")]:
    ax.plot(x[i], serie.values[i], "o", markersize=9, color=cor,
            markeredgecolor=PALETA["surface"], markeredgewidth=2)
    ax.annotate(f"{txt}: {x[i].strftime('%b/%y')}", (x[i], serie.values[i]),
                textcoords="offset points", xytext=(0, dy), ha="center",
                color=cor, fontsize=10, fontweight="bold")

ax.yaxis.set_major_formatter(mticker.FuncFormatter(brl))
ax.set_ylim(0, serie.max() * 1.18)
ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 4, 7, 10]))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b/%y"))
ax.set_ylabel("Receita liquida")
acabamento(ax, "Receita mensal ao longo de 24 meses",
           "Sazonalidade clara: novembro e dezembro puxam o ano; janeiro e o fundo do poco")
plt.show()

**Regra de ouro do gráfico de linha:** o eixo Y começa no zero (`set_ylim(0, ...)`).
Cortar a base do eixo é a forma mais fácil de exagerar uma variação sem mentir nos números.

#### 8.3 Duas séries — comparar dois anos

Quando há mais de uma série, a legenda é obrigatória. Aqui vai legenda **e** rótulo direto
no fim de cada linha: quem olha rápido não precisa ir e voltar até a legenda.

In [ ]:
p = df.pivot_table(index=df["Data_Venda"].dt.month, columns=df["Data_Venda"].dt.year,
                   values="Valor_Venda", aggfunc="sum")
meses = ["Jan", "Fev", "Mar", "Abr", "Mai", "Jun", "Jul", "Ago", "Set", "Out", "Nov", "Dez"]
cores = [PALETA["azul"], PALETA["laranja"]]

fig, ax = plt.subplots(figsize=(8, 4.4))
for k, ano in enumerate(p.columns):
    ax.plot(p.index, p[ano].values, color=cores[k], linewidth=2, marker="o", markersize=6,
            markeredgecolor=PALETA["surface"], markeredgewidth=1.5, label=str(ano))
    ax.annotate(str(ano), (p.index[-1], p[ano].values[-1]), textcoords="offset points",
                xytext=(8, 0), va="center", color=cores[k], fontsize=10, fontweight="bold")

ax.set_xticks(range(1, 13)); ax.set_xticklabels(meses); ax.set_xlim(0.6, 12.9)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(brl))
ax.set_ylabel("Receita liquida")
ax.legend(loc="upper left", labelcolor=PALETA["ink2"])
acabamento(ax, "Receita por mes: 2024 vs 2025",
           "Mesmo formato de sazonalidade nos dois anos, com patamar maior em 2025")
plt.show()

#### 8.4 Barra horizontal — ranking

Barra **horizontal**, não vertical: nome de produto não cabe deitado no eixo X.
E sempre ordenada — um ranking fora de ordem obriga o leitor a fazer o trabalho.

In [ ]:
top = df.groupby("Subproduto")["Valor_Venda"].sum().nlargest(10).sort_values()

fig, ax = plt.subplots(figsize=(8, 5.2))
barras = ax.barh(top.index, top.values, color=PALETA["azul"], height=0.72)

for b, v in zip(barras, top.values):
    ax.text(v * 1.01, b.get_y() + b.get_height() / 2, brl(v),
            va="center", color=PALETA["ink2"], fontsize=9.5)

ax.xaxis.set_major_formatter(mticker.FuncFormatter(brl))
ax.set_xlim(0, top.max() * 1.22)
ax.tick_params(axis="y", labelcolor=PALETA["ink2"])
acabamento(ax, "Top 10 subprodutos por receita",
           "Concentracao em itens de ticket alto: eletronicos e eletrodomesticos dominam", grid="x")
plt.show()

**Uma cor só.** Colorir cada barra de um tom diferente parece mais bonito, mas o
comprimento já diz quem é maior — a cor variando não acrescenta informação e ainda sugere
que as barras pertencem a grupos diferentes.

#### 8.5 Boxplot — distribuição comparada entre grupos

O histograma mostra uma distribuição. O boxplot mostra **várias lado a lado**. A caixa vai
do 1º ao 3º quartil (o miolo dos 50%), a linha no meio é a mediana, e os pontos soltos são
os outliers.

In [ ]:
ordem = df.groupby("Categoria")["Margem_Pct"].median().sort_values().index.tolist()
dados = [df.loc[df["Categoria"] == c, "Margem_Pct"].values * 100 for c in ordem]

fig, ax = plt.subplots(figsize=(8, 4.6))
bp = ax.boxplot(dados, vert=False, patch_artist=True, widths=0.55,
                medianprops=dict(color=PALETA["surface"], linewidth=2),
                whiskerprops=dict(color=PALETA["axis"], linewidth=1.2),
                capprops=dict(color=PALETA["axis"], linewidth=1.2),
                flierprops=dict(marker="o", markersize=3.5, markerfacecolor=PALETA["muted"],
                                markeredgecolor="none", alpha=0.35))
for caixa in bp["boxes"]:
    caixa.set_facecolor(PALETA["azul"]); caixa.set_edgecolor(PALETA["surface"]); caixa.set_linewidth(1.5)

ax.set_yticks(range(1, len(ordem) + 1)); ax.set_yticklabels(ordem)
ax.axvline(0, color=PALETA["laranja"], linewidth=1.5, linestyle="--")
ax.text(0.6, 5.42, "margem zero", color=PALETA["laranja"], fontsize=9.5, fontweight="bold")
ax.set_ylim(0.3, 5.9)
ax.xaxis.set_major_formatter(mticker.PercentFormatter())
ax.set_xlabel("Margem bruta (%)")
ax.tick_params(axis="y", labelcolor=PALETA["ink2"])
acabamento(ax, "Distribuicao da margem por categoria",
           "Vestuario tem a maior margem; eletronicos puxam as vendas no prejuizo", grid="x")
plt.show()

**Ordene os grupos pela mediana**, nunca em ordem alfabética. Ordem alfabética é uma
informação sobre o alfabeto, não sobre o negócio.

#### 8.6 Dispersão — duas medidas andam juntas?

In [ ]:
amostra = df.sample(2500, random_state=7)

fig, ax = plt.subplots(figsize=(8, 4.6))
ax.scatter(amostra["Desconto_Pct"] * 100, amostra["Margem_Pct"] * 100,
           s=16, color=PALETA["azul"], alpha=0.35, edgecolors="none")

coef = np.polyfit(df["Desconto_Pct"] * 100, df["Margem_Pct"] * 100, 1)
xs = np.linspace(0, df["Desconto_Pct"].max() * 100, 50)
ax.plot(xs, np.polyval(coef, xs), color=PALETA["laranja"], linewidth=2.5)
ax.text(0.985, 0.955,
        f"linha de tendencia\ncada +1 p.p. de desconto\nderruba {abs(coef[0]):.2f} p.p. de margem",
        transform=ax.transAxes, ha="right", va="top",
        color=PALETA["laranja"], fontsize=10, fontweight="bold", linespacing=1.45)

ax.axhline(0, color=PALETA["muted"], linewidth=1, linestyle="--")
ax.xaxis.set_major_formatter(mticker.PercentFormatter(decimals=0))
ax.yaxis.set_major_formatter(mticker.PercentFormatter(decimals=0))
ax.set_xlabel("Desconto concedido"); ax.set_ylabel("Margem bruta")
acabamento(ax, "Desconto x margem",
           "Acima de 25% de desconto, 48% das vendas ficam com margem negativa")
plt.show()

`alpha=0.35` deixa os pontos semitransparentes: onde eles se acumulam, a mancha fica mais
escura. Sem isso, 2.500 pontos viram um borrão sólido e você perde a densidade.

E o número que fecha a análise:

In [ ]:
faixas = pd.cut(df["Desconto_Pct"] * 100, [-0.1, 5, 10, 15, 20, 25, 40],
                labels=["0-5%", "5-10%", "10-15%", "15-20%", "20-25%", "25%+"])
(df.assign(negativa=df["Margem_Pct"] < 0)
   .groupby(faixas, observed=True)["negativa"].mean()
   .mul(100).round(1).to_frame("% de vendas com margem negativa"))

### seaborn: o atalho

Para gráfico estatístico exploratório, o seaborn faz em uma linha o que no matplotlib
levaria dez. A troca é controle fino sobre o resultado — por isso o rascunho costuma ser
seaborn e o gráfico final, matplotlib.

In [ ]:
import seaborn as sns
sns.boxplot(data=df, y="Categoria", x="Margem_Pct", color=PALETA["azul"])
plt.show()

---
## Módulo 9 — Exportar o resultado

Nada disso serve se o resultado não sai do notebook.

In [ ]:
painel_final = (
    df[df["Status_Pedido"] == "Concluida"]
      .groupby(["Regiao", "Categoria"])
      .agg(pedidos=("ID_Venda", "count"),
           receita=("Valor_Venda", "sum"),
           margem=("Margem_Valor", "sum"))
      .assign(margem_pct=lambda d: d["margem"] / d["receita"])
      .reset_index()
      .sort_values("receita", ascending=False)
)

painel_final.to_excel("resumo_por_regiao.xlsx", index=False)
painel_final.to_csv("resumo_por_regiao.csv", index=False, sep=";", decimal=",", encoding="utf-8-sig")

print("arquivos gravados")
painel_final.head()

**Para o CSV que vai abrir no Excel brasileiro:** `sep=";"`, `decimal=","` e
`encoding="utf-8-sig"`. Sem o `utf-8-sig`, os acentos viram caracteres estranhos.
Sem `index=False`, você exporta uma coluna extra com o índice.

Para gravar várias abas em um mesmo arquivo:

In [ ]:
with pd.ExcelWriter("relatorio.xlsx") as writer:
    painel_final.to_excel(writer, sheet_name="Regiao x Categoria", index=False)
    resumo.reset_index().to_excel(writer, sheet_name="Resumo Regiao", index=False)
    painel.to_excel(writer, sheet_name="Atingimento", index=False)

print("relatorio.xlsx com 3 abas")

---
## Módulo 10 — Exercícios

Sem consultar as respostas. Cada um usa só o que está acima.

1. Qual **vendedor** teve a maior margem percentual em 2025, considerando apenas pedidos
   concluídos e com pelo menos 200 vendas?
2. Monte uma tabela com **canal de venda nas linhas e trimestre nas colunas**, com a
   receita em cada célula.
3. Quantos **clientes distintos** compraram nos dois anos (2024 **e** 2025)?
   *Dica: `set` ou `merge` com `indicator=True`.*
4. Qual **categoria** mais perdeu margem percentual de 2024 para 2025?
5. Faça um gráfico de barras horizontais com a **receita por forma de pagamento**,
   ordenado, com o valor rotulado em cada barra.
6. Existe alguma **região** onde o desconto médio é significativamente maior que nas
   outras? Mostre com um boxplot.

### Erros que valem conhecer antes de cometer

| O erro | O que acontece | Como evitar |
|---|---|---|
| `merge` sem `validate` | Chave duplicada multiplica linhas e infla a receita | `validate="many_to_one"` |
| Usar `and` no lugar de `&` | `ValueError: truth value is ambiguous` | Sempre `&` e `\|`, com parênteses |
| `SettingWithCopyWarning` | Você editou uma cópia; o original não mudou | Use `.loc[linhas, coluna] = valor` |
| Média em distribuição assimétrica | O número não descreve ninguém | Olhe o histograma antes; use mediana |
| Eixo Y que não começa no zero | Exagera a variação | `ax.set_ylim(0, ...)` |
| `groupby` e esquecer `reset_index` | A chave fica no índice e nada funciona depois | `.reset_index()` |

### Para onde ir depois

1. **`.pipe()` e cadeias longas** — escrever análise como uma sequência legível.
2. **`merge_asof`** — junção por data aproximada, essencial em base de crédito.
3. **Janelas: `rolling`, `expanding`, `shift`** — média móvel, acumulado, comparação com período anterior.
4. **Parquet no lugar de CSV** — `df.to_parquet()`, muito mais rápido e preserva os tipos.
5. **scikit-learn** — quando a pergunta deixa de ser "o que aconteceu" e vira "o que vai acontecer".

---
## Cola de bolso: SQL → pandas

| SQL | pandas |
|---|---|
| `SELECT a, b FROM t` | `df[["a", "b"]]` |
| `SELECT *` | `df` |
| `SELECT DISTINCT a` | `df["a"].unique()` |
| `WHERE a = 1 AND b > 2` | `df[(df["a"] == 1) & (df["b"] > 2)]` |
| `WHERE a IN ('x','y')` | `df[df["a"].isin(["x", "y"])]` |
| `WHERE a LIKE '%x%'` | `df[df["a"].str.contains("x")]` |
| `WHERE a IS NULL` | `df[df["a"].isna()]` |
| `ORDER BY a DESC` | `df.sort_values("a", ascending=False)` |
| `LIMIT 10` | `df.head(10)` |
| `TOP 10 BY a` | `df.nlargest(10, "a")` |
| `COUNT(*)` | `len(df)` ou `df.shape[0]` |
| `COUNT(DISTINCT a)` | `df["a"].nunique()` |
| `GROUP BY a` | `df.groupby("a")` |
| `SUM(x) GROUP BY a` | `df.groupby("a")["x"].sum()` |
| `HAVING SUM(x) > 100` | `s = df.groupby("a")["x"].sum(); s[s > 100]` |
| `LEFT JOIN ON a` | `df.merge(outra, on="a", how="left")` |
| `UNION ALL` | `pd.concat([df1, df2])` |
| `CASE WHEN ... END` | `np.select(condicoes, resultados, default=...)` |
| `DATE_TRUNC('month', d)` | `df["d"].dt.to_period("M")` |
| `LAG(x, 12)` | `df["x"].shift(12)` |
| `ROW_NUMBER() OVER (...)` | `df.groupby("a").cumcount() + 1` |
| `CREATE TABLE AS ...` | `df.to_excel(...)` / `df.to_parquet(...)` |